In [0]:
from pyspark.sql import functions as F, Window

spark.sql("USE CATALOG ecommerce_project")
spark.sql("USE SCHEMA sales")

In [0]:
%run ./00_transformations

In [0]:
enriched_customers = clean_customers(spark.table("raw_customers"))
enriched_products = clean_products(spark.table("raw_products"))

save_table(enriched_customers, "enriched_customers", expected_min_rows=793)
save_table(enriched_products, "enriched_products", expected_min_rows=1818)


In [0]:
print("customers:", spark.table("enriched_customers").count())   # expect 793
print("products :", spark.table("enriched_products").count())    # expect 1818 (1851 - 33)
display(spark.table("enriched_customers").select("customer_id", "customer_name").limit(10))
display(spark.table("enriched_customers").filter("customer_name IS NULL"))

## Debug note 1 — raw_products: mis-parsed CSV, price came back as string

**Symptom:** `clean_products` failed on `.cast("double")` for `price_per_product` with
`CAST_INVALID_INPUT: 'New York' ... cannot be cast to "DOUBLE"`.

**Where I noticed it:** Back in Step 2, `raw_products.printSchema()` already showed
`price_per_product: string`, even though `inferSchema` was on for that CSV read. That was
the actual warning sign — a numeric column inferred as string usually means the reader
mis-split a row.

**Root cause:** The original read of `Products.csv` had no `quote`/`escape` options:
`spark.read.option("header","true").option("inferSchema","true").csv(...)`. Product names
containing a literal comma or a quote character (e.g. `"Leather Task Chair, Black"`,
`6" Cubicle Wall Clock`) got split on the embedded comma, shifting every column after
`product_name` by one. 52 rows ended up with a US state name sitting in the price column.

**How I confirmed it:** Ran a throwaway (non-writing) cell that re-read the CSV with
`multiLine=true`, `quote='"'`, `escape='"'`, and compared `try_cast(... as double) IS NULL`
counts: 52 bad rows with the old read, 0 with the fixed read.

**Fix:** Added `.option("multiLine","true").option("quote",'"').option("escape",'"')` to the
products read in `01_raw_ingestion`, and `.option("overwriteSchema","true")` on the
`saveAsTable` call, since the column type changes from string to double and Delta blocks a
schema change without that flag.

**Verified:** `raw_products` count unchanged at 1851; 0 rows with non-numeric price;
`price_per_product` schema is now `double`; 147 rows have a real embedded quote in
`product_name` (e.g. `6" Cubicle Wall Clock`) and now stay intact as one field.

## Debug note 2 — Customer.csv never arrived; switched to Customer.xlsx with pure PySpark

**Symptom:** Step 2's plan assumed a `Customer.csv` would be uploaded to the `raw_files`
volume. Listing the volume showed only `Customer.xlsx`, `Orders.json`, and `Products.csv`
— no CSV.

**First fix attempted (rejected on review):** Read `Customer.xlsx` with `pandas.read_excel`,
converted to a Spark DataFrame with `spark.createDataFrame(pdf)`. The user's standard is
pure PySpark only — pandas isn't allowed in this pipeline — so this was replaced.

**Final fix:** Dropped pandas entirely. Used Spark's built-in Excel data source instead:
`spark.read.format("excel").option("headerRows", 1).load(...)`, then cast every column to
string so the schema matches what the CSV path would have produced (all columns as text,
consistent with Step 2's intent for `raw_customers`).

**Verified:** `raw_customers` row count is 793 both with the pandas version and the pure
PySpark version — same data, same result, no pandas dependency.

## Debug note 3 — Step 3 enrichment: two expected-but-worth-checking drops (not bugs)

**33 duplicate `product_id` values in `raw_products`.** `clean_products` keeps exactly one
row per `product_id` using a `row_number()` window ordered by `state`, `product_name` — a
deterministic tie-break, same winner every run. Confirmed with a standalone groupBy/count
query on `raw_products`: 33 ids appear more than once, 33 extra rows total, so
1851 - 33 = 1818 matches the expected `enriched_products` count exactly. This is real
source duplication (the same product listed under more than one state row), not a bug
introduced by the fix in note 1 — the duplicate count didn't change before/after that fix.

**8 rows with `customer_name IS NULL` in `enriched_customers`.** Checked the 8 affected
rows directly in `raw_customers`: the `customer_name` field was already empty in the
source Excel file for all 8. `clean_customers` didn't null anything — there was nothing to
clean. 8 customers have no name in the source file, so `customer_name` is null. The
customer IDs and orders are kept. This is expected, not a bug.

**Extra finding — inconsistent phone values (not a bug, just noted):** 135 customers in
`raw_customers` have unusual `phone` values (formats like `-7269`, `#ERROR!`, and other
non-phone-shaped strings), not only the 2 that showed up in the null-name rows above. Phone
isn't used in any required output, so it was left as is. Phone values are inconsistent in
format and not used in any output, so they were not standardized.

**Net result:** both counts match the targets given — `enriched_customers`: 793,
`enriched_products`: 1818 — with the 33/8 "losses" explained and intentional, not silent
data loss.